In [24]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Carregar dataset
dos_path = Path("../data/processed/DoS_with_gap_features.csv")
df = pd.read_csv(dos_path)

print(f"Shape original: {df.shape}")

# 🔴 Remover duplicatas ANTES de qualquer processamento
df = df.drop_duplicates()

print(f"Shape após remover duplicatas: {df.shape}")

Shape original: (94625, 67)
Shape após remover duplicatas: (94007, 67)


In [14]:
df = df.drop_duplicates()

In [15]:
num_df = df.select_dtypes(include=['int64', 'float64'])

colunas_para_remover = [
    'frame.time_delta_displayed', 'frame.time_epoch', 'frame.time_invalid', 
    'frame.time_relative', 'tcp.srcport', 'tcp.dstport', 
    'frame.coloring_rule.name', 'frame.coloring_rule.string', 
    'frame.comment', 'frame.comment.expert', 'frame.encap_type', 
    'frame.file_off', 'frame.ignored', 'frame.incomplete', 
    'frame.interface_id', 'frame.interface_name', 'frame.link_nr', 
    'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift',
    'mqtt.msgid', 'mqtt.username', 'mqtt.passwd', 
    'mqtt.willmsg', 'mqtt.willtopic'
]

colunas_existentes = [c for c in colunas_para_remover if c in num_df.columns]

X = num_df.drop(columns=colunas_existentes, errors='ignore')

In [16]:
X = X.fillna(0)
X = X.replace([np.inf, -np.inf], 0)

In [19]:
def remover_outliers_iqr_colunas(df, colunas, threshold=1.5):
    df_clean = df.copy()
    
    mask = pd.Series(True, index=df_clean.index)
    
    for col in colunas:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR
        
        mask &= (df_clean[col] >= lower) & (df_clean[col] <= upper)
    
    return df_clean[mask]

In [20]:
# Definir colunas críticas (baseado no seu describe)
colunas_criticas = ['publish_gap', 'connect_gap']

# Aplicar remoção de outliers
X_sem_outliers = remover_outliers_iqr_colunas(X, colunas_criticas)

# Pegar índices válidos
indices_validos = X_sem_outliers.index

# Atualizar X
X = X_sem_outliers

# Atualizar df (alinhamento correto)
df = df.loc[indices_validos]

In [21]:
le = LabelEncoder()
y = le.fit_transform(df['type'])

In [22]:
print("X:", X.shape)
print("df:", df.shape)
print("y:", len(y))

X: (45633, 29)
df: (45633, 67)
y: 45633


In [23]:
df_limpo = df.copy()

# Substituir target por versão numérica
df_limpo['type'] = y

df_limpo.to_csv('../data/processed/DoS_clean.csv', index=False)

In [25]:
# === LIMPEZA ESPECÍFICA: DoS_with_gap_features.csv ===
# Carregar arquivo original
dos_with_gap_path = Path('../data/processed/DoS_with_gap_features.csv')
df_dos_gap = pd.read_csv(dos_with_gap_path)

print(f"\n{'='*60}")
print(f"LIMPEZA: DoS_with_gap_features.csv")
print(f"{'='*60}")
print(f"Shape original: {df_dos_gap.shape}")

# Remover duplicatas
df_dos_gap = df_dos_gap.drop_duplicates()
print(f"Shape após remover duplicatas: {df_dos_gap.shape}")

# Definir colunas a remover
colunas_remover = [
    'frame.time_delta_displayed', 'frame.time_epoch', 'frame.time_invalid', 
    'frame.time_relative', 'tcp.srcport', 'tcp.dstport', 
    'frame.coloring_rule.name', 'frame.coloring_rule.string', 
    'frame.comment', 'frame.comment.expert', 'frame.encap_type', 
    'frame.file_off', 'frame.ignored', 'frame.incomplete', 
    'frame.interface_id', 'frame.interface_name', 'frame.link_nr', 
    'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift',
    'mqtt.msgid', 'mqtt.username', 'mqtt.passwd', 
    'mqtt.willmsg', 'mqtt.willtopic'
]

# Remover apenas as que existem
colunas_removidas = [c for c in colunas_remover if c in df_dos_gap.columns]
df_dos_gap = df_dos_gap.drop(columns=colunas_removidas, errors='ignore')

print(f"Shape após remover {len(colunas_removidas)} colunas: {df_dos_gap.shape}")
print(f"\nColunas removidas ({len(colunas_removidas)}):")
print(f"  {colunas_removidas}")

# === REMOVER OUTLIERS ===
print(f"\n{'─'*60}")
print("REMOVENDO OUTLIERS (Método: IQR com threshold=1.5)")
print(f"{'─'*60}")

# Selecionar apenas colunas numéricas para análise de outliers
colunas_numericas = df_dos_gap.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Colunas numéricas para análise: {len(colunas_numericas)}")

# Aplicar função de remoção de outliers com IQR
def remover_outliers_iqr(df, colunas, threshold=1.5):
    df_clean = df.copy()
    mask = pd.Series(True, index=df_clean.index)
    
    for col in colunas:
        Q1 = df_clean[col].quantile(0.25)
        Q3 = df_clean[col].quantile(0.75)
        IQR = Q3 - Q1
        
        lower = Q1 - threshold * IQR
        upper = Q3 + threshold * IQR
        
        mask &= (df_clean[col] >= lower) & (df_clean[col] <= upper)
    
    return df_clean[mask]

# Remover outliers
shape_antes = df_dos_gap.shape[0]
df_dos_gap = remover_outliers_iqr(df_dos_gap, colunas_numericas, threshold=1.5)
shape_depois = df_dos_gap.shape[0]
removidos = shape_antes - shape_depois

print(f"Linhas antes: {shape_antes}")
print(f"Linhas após remover outliers: {shape_depois}")
print(f"Linhas removidas: {removidos} ({100*removidos/shape_antes:.2f}%)")

# Salvar arquivo limpo
output_path = Path('../data/processed/DoS_with_gap_features_clean.csv')
df_dos_gap.to_csv(output_path, index=False)
print(f"\n✓ Arquivo salvo em: {output_path}")
print(f"✓ Shape final: {df_dos_gap.shape}")
print(f"✓ Colunas finais ({len(df_dos_gap.columns)}): {list(df_dos_gap.columns)}")


LIMPEZA: DoS_with_gap_features.csv
Shape original: (94625, 67)
Shape após remover duplicatas: (94007, 67)
Shape após remover 26 colunas: (94007, 41)

Colunas removidas (26):
  ['frame.time_delta_displayed', 'frame.time_epoch', 'frame.time_invalid', 'frame.time_relative', 'tcp.srcport', 'tcp.dstport', 'frame.coloring_rule.name', 'frame.coloring_rule.string', 'frame.comment', 'frame.comment.expert', 'frame.encap_type', 'frame.file_off', 'frame.ignored', 'frame.incomplete', 'frame.interface_id', 'frame.interface_name', 'frame.link_nr', 'frame.marked', 'frame.md5_hash', 'frame.number', 'frame.offset_shift', 'mqtt.msgid', 'mqtt.username', 'mqtt.passwd', 'mqtt.willmsg', 'mqtt.willtopic']

────────────────────────────────────────────────────────────
REMOVENDO OUTLIERS (Método: IQR com threshold=1.5)
────────────────────────────────────────────────────────────
Colunas numéricas para análise: 29
Linhas antes: 94007
Linhas após remover outliers: 0
Linhas removidas: 94007 (100.00%)

✓ Arquivo sa

In [3]:
# Carregar o dataset DoS
dos_path = Path("../data/processed/DoS_with_gap_features_clean.csv")
df = pd.read_csv(dos_path)

print(f"📊 Shape do dataset: {df.shape}")
print(f"📝 Colunas: {df.shape[1]}")
print(f"📋 Registros: {df.shape[0]}")
print(f"\n🏷️ Distribuição do Label ('type'):")
print(df['type'].value_counts())

NameError: name 'Path' is not defined